# Newton's Form and the Divided-Difference Table

Through $n$ points with distinct $x$ values there is exactly one polynomial of degree less than $n$. The polynomial is unique, but the way we write it down is not, and that choice matters more than you might expect. It decides how much arithmetic an evaluation costs, and how much work it takes to add one more data point.

The Lagrange form writes the interpolant as $\sum_i y_i \ell_i(x)$, where the coefficients are handed to us for free. Add a node, though, and every basis polynomial $\ell_i$ has to be rebuilt from scratch. Newton's form pays a little more up front and gets it back later: the coefficients drop out of a triangular table, and appending a node adds exactly one term while leaving every earlier one alone.

Let's build that table, then find a cheap way to evaluate what it gives us.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
# Enable interactive ipywidgets sliders. try/except so the notebook also runs
# in plain Jupyter (outside Colab) without error.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown, Checkbox

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Divided differences

Newton's form builds the interpolant one node at a time. Start with $p_0(x) = c_0$ and set $c_0 = f(x_0)$. At each later step, add a term that vanishes at every node already matched:

$$ p_k(x) = p_{k-1}(x) + c_k \prod_{j=0}^{k-1}(x - x_j). $$

The new term is zero at $x_0,\dots,x_{k-1}$, so the earlier data stay interpolated and the single condition $p_k(x_k) = f(x_k)$ pins down $c_k$. Solve for the coefficients and they come out as

$$ p_n(x) = \sum_{k=0}^{n} f[x_0,\dots,x_k]\prod_{j=0}^{k-1}(x - x_j), \qquad f[x_i,\dots,x_{i+j}] = \frac{f[x_{i+1},\dots,x_{i+j}] - f[x_i,\dots,x_{i+j-1}]}{x_{i+j}-x_i}. $$

The quantities $f[x_i,\dots,x_{i+j}]$ are called **divided differences**, and the recurrence says each one is assembled from two of one lower order. Collect them in a triangular table and the coefficients we want, $f[x_0,\dots,x_k]$, sit along the top row. Append a node and the table gains one anti-diagonal while nothing already computed moves (why?).

In [ ]:
# ---------------------------------------------------------------------------
# The divided-difference table
# ---------------------------------------------------------------------------
# Newton's form writes the interpolating polynomial as
#
#   p(x) = c0 + c1 (x-x0) + c2 (x-x0)(x-x1) + ... + cn (x-x0)...(x-x_{n-1})
#
# and the coefficients c_k come out of the triangular table built here.

def divided_differences(x, y):
    """Build the divided-difference table for data (x[i], y[i]).

    Parameters
    ----------
    x : array (n,)  distinct nodes
    y : array (n,)  values, y[i] = f(x[i])

    Returns
    -------
    coeffs : array (n,)   Newton coefficients c_0 .. c_{n-1}
    table  : array (n,n)  the triangular table; unwritten entries stay zero
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    table = np.zeros((n, n))
    table[:, 0] = y
    for j in range(1, n):
        for i in range(n - j):
            table[i, j] = (table[i + 1, j - 1] - table[i, j - 1]) / (x[i + j] - x[i])
    coeffs = table[0, :].copy()
    return coeffs, table

In [ ]:
# ---------------------------------------------------------------------------
# A readable printout of the triangular table
# ---------------------------------------------------------------------------
def print_dd_table(x, table):
    """Pretty-print the divided-difference table as a triangle.

    Row i, column j holds f[x_i, ..., x_{i+j}].
    """
    n = len(x)
    print("  x_i   |  f[.] (order increases left -> right)")
    print("-" * 60)
    for i in range(n):
        row = f"{x[i]:6.3f} |"
        for j in range(n - i):
            row += f" {table[i, j]:10.4f}"
        print(row)

In [ ]:
# ---------------------------------------------------------------------------
# The table for a small data set
# ---------------------------------------------------------------------------
x_data = [0.0, 1.0, 3.0, 6.0]
y_data = [1.0, 4.0, 2.0, 8.0]

coeffs, table = divided_differences(x_data, y_data)
print_dd_table(x_data, table)
print()
print("Newton coefficients c0..c3:", np.array2string(coeffs, precision=4))

## Evaluating the Newton form

We have the polynomial, but we still can't get a number out of it. Written out term by term,

$$ p(x) = c_0 + c_1(x-x_0) + c_2(x-x_0)(x-x_1) + \cdots, $$

the $k$th term needs $k$ multiplications, so the whole sum costs $\mathcal{O}(n^2)$ at every point we ask about. That's wasteful, since each term rebuilds a product the term before it already had. What if we factor the shared pieces out? The sum nests:

$$ p(x) = c_0 + (x-x_0)\Bigl(c_1 + (x-x_1)\bigl(c_2 + \cdots + (x-x_{n-2})\,c_{n-1}\bigr)\Bigr). $$

Read it from the inside out: start from the last coefficient, then repeatedly multiply by $(x - x_k)$ and add $c_k$, for $k = n-2$ down to $0$. Each step is one multiplication and one addition, so the cost per point drops to $\mathcal{O}(n)$. This is Horner's rule, applied to the Newton basis instead of to the powers of $x$, and it's steadier in floating point as well as cheaper, because it never forms the long products that the term-by-term sum has to.

In [ ]:
# ---------------------------------------------------------------------------
# Nested evaluation
# ---------------------------------------------------------------------------
def newton_eval(x_nodes, coeffs, xq):
    """Evaluate the Newton-form polynomial at the query points xq.

    Parameters
    ----------
    x_nodes : array (n,)  interpolation nodes, in the order used to build coeffs
    coeffs  : array (n,)  Newton coefficients from divided_differences
    xq      : array       query points

    Returns
    -------
    array   the interpolant evaluated at xq
    """
    xq = np.asarray(xq, dtype=float)
    n = len(coeffs)
    result = np.full_like(xq, coeffs[n - 1])
    for k in range(n - 2, -1, -1):
        result = result * (xq - x_nodes[k]) + coeffs[k]
    return result

In [ ]:
# ---------------------------------------------------------------------------
# The interpolant through the data, and a check that it interpolates
# ---------------------------------------------------------------------------
for t in [0.5, 2.0, 5.0]:
    print(f"p({t}) = {newton_eval(x_data, coeffs, t):.6f}")

print()
print("at the nodes, p reproduces the data:")
print("p(x_i) =", np.array2string(newton_eval(x_data, coeffs, np.array(x_data)), precision=4))
print("y_i    =", np.array2string(np.asarray(y_data), precision=4))

In [ ]:
# ---------------------------------------------------------------------------
# Incremental construction
# ---------------------------------------------------------------------------
# Append one more node to the end of the list and only one new coefficient
# appears; the earlier ones are unchanged, since c_k depends only on x_0..x_k.
# (Lagrange, by contrast, must rebuild every basis polynomial from scratch.)
# The reuse needs the old nodes to survive in the same order. Rebuilding an
# equally spaced set at each size moves every node, and then every coefficient
# changes. The cell after this one shows both cases side by side.

def f_demo(x):
    """Sample function, smooth and oscillatory."""
    return np.sin(2 * x) + 0.5 * x

def show_newton(n_nodes=5, show_table=True, a=0.0, b=5.0):
    """Sample f_demo at n_nodes equally spaced points, build the Newton
    interpolant, plot it against f, and optionally print the table + coeffs."""
    x = np.linspace(a, b, n_nodes)           # nodes
    y = f_demo(x)                            # data values
    coeffs, table = divided_differences(x, y)

    xx = np.linspace(a, b, 400)
    p = newton_eval(x, coeffs, xx)

    plt.figure()
    plt.plot(xx, f_demo(xx), "k-", lw=2, label="f(x)")
    plt.plot(xx, p, "r--", lw=2, label=f"Newton interpolant (n={n_nodes})")
    plt.plot(x, y, "bo", ms=7, label="data")
    plt.legend(); plt.xlabel("x"); plt.title("Newton-form interpolation")
    plt.show()

    if show_table:
        print_dd_table(x, table)
        print("\nNewton coefficients c0..c{}:".format(n_nodes - 1))
        print(np.array2string(coeffs, precision=4))
        print("\nNote. These nodes are rebuilt as an equally spaced set at every "
              "size, so\nevery coefficient moves when n changes. See the next cell "
              "for the appended case.")

show_newton(5)

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: slide the number of nodes; toggle the table
# ---------------------------------------------------------------------------
interact(
    show_newton,
    n_nodes=IntSlider(min=2, max=12, step=1, value=5, description="# nodes"),
    show_table=Checkbox(value=True, description="show table"),
    a=(-1.0, 2.0, 0.5), b=(3.0, 8.0, 0.5),
);

In [ ]:
# ---------------------------------------------------------------------------
# Appended nodes versus a rebuilt equally spaced set
# ---------------------------------------------------------------------------
def coeffs_for(nodes):
    """Newton coefficients for f_demo sampled at these nodes, in this order."""
    nodes = np.asarray(nodes, dtype=float)
    c, _ = divided_differences(nodes, f_demo(nodes))
    return c

appended = [0.0, 2.5, 5.0, 1.25, 3.75, 0.625, 1.875, 3.125, 4.375]
print("each new node appended to the end of the list")
for m in [3, 5, 9]:
    print(f"  m = {m}  {np.array2string(coeffs_for(appended[:m]), precision=4)}")

print("\nan equally spaced set rebuilt from scratch at each size")
for m in [3, 5, 9]:
    print(f"  m = {m}  {np.array2string(coeffs_for(np.linspace(0, 5, m)), precision=4)}")

## Summary

Newton and Lagrange are two ways of writing the same unique polynomial. Building the divided-difference table costs $\mathcal{O}(n^2)$, and evaluating by nested multiplication costs $\mathcal{O}(n)$ per point. Appending a node at the end of the list adds exactly one coefficient and reuses every earlier one, though reordering the nodes or replacing one doesn't.

## Things to try

- Raise the node count one step at a time. The slider rebuilds an equally spaced set, so every coefficient moves. Compare that with the appended-node cell above, where the earlier entries stay put. What is the difference between the two cases?
- Edit the node array to put two nodes almost on top of each other, then watch the high-order columns of the table. The recurrence divides by $x_{i+j}-x_i$, so what should happen to those entries as the gap shrinks?
- Push the node count up to 12 and Runge's phenomenon shows up. It doesn't care which representation of the polynomial you chose. Would Chebyshev nodes rescue it here?